# Gráficos del informe interno

Genera las figuras de `docs/informe_interno.md` a partir de los archivos del repo (`resultados/`, `data/` y `archivo/resultados/`). Cada figura se guarda en `docs/figuras/` como PNG.

**En Colab:** *Ejecutar todas*. El notebook trae el repo de GitHub solo para leer los resultados (no usa GPU ni Google Drive).

Los datos que no están en archivos del repo (pérdida de validación de los mT5, calibración de la regla de votos y resultado de las muestras de revisión) están escritos en la celda correspondiente, con su fuente en el informe.

## 1. Preparación

In [ ]:
import os
import re
import subprocess
import unicodedata

import matplotlib

matplotlib.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAMA = "entrega-final"
URL_REPO = "https://github.com/carlosiskindacool/proyecto-nlp.git"

# En Colab se clona el repo; si ya se corre dentro del repo (o de notebooks/), se usa tal cual.
if not os.path.exists("resultados"):
    if os.path.exists("../resultados"):
        os.chdir("..")
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", RAMA, URL_REPO, "repo"], check=True)
        os.chdir("repo")
os.makedirs("docs/figuras", exist_ok=True)
print("carpeta de trabajo:", os.getcwd())

# Nombres de los sistemas: en el código y los CSV `reglas_curado` se llama `reglas_flo`, etc.
NOMBRE = {"identidad": "identidad", "reglas_equipo": "reglas_equipo", "reglas_flo": "reglas_curado", "mt5_flo_fp32": "mT5 denso",
          "mt5": "mT5 denso", "mt5_moe_fp32": "mT5 con MoE", "mt5_moe": "mT5 con MoE", "llm_zero_shot": "LLM zero-shot",
          "llm_dicc_flo": "LLM con diccionario"}
COLOR = {"identidad": "#CFC7BF", "reglas_equipo": "#8E8A86", "reglas_curado": "#FF946C", "mT5 denso": "#9B6BC4", "mT5 con MoE": "#5B3E96",
         "LLM zero-shot": "#6FA8C8", "LLM con diccionario": "#2E6F95"}


def guardar(fig, nombre):
    fig.tight_layout()
    fig.savefig(f"docs/figuras/{nombre}.png", dpi=160, bbox_inches="tight", facecolor="white")
    plt.show()
    print("guardada docs/figuras/%s.png" % nombre)


def barras(ax, tabla, colores=None, ancho=0.8):
    """tabla: DataFrame con filas = grupos (eje x) y columnas = sistemas."""
    n = len(tabla.columns)
    x = np.arange(len(tabla))
    for i, col in enumerate(tabla.columns):
        b = ax.bar(x + (i - (n - 1) / 2) * ancho / n, tabla[col].values, ancho / n, label=col, color=(colores or COLOR).get(col, None))
        ax.bar_label(b, fmt="%.1f", fontsize=7, padding=1)
    ax.set_xticks(x)
    ax.set_xticklabels(tabla.index)

## 2. Fase 1: estudio previo (BLEU)

BLEU de seis de los ocho sistemas en los tres conjuntos de la fase 1. Se omiten los traductores genéricos (BLEU de 25 a 50) para no aplastar la escala. La identidad es el piso.

In [ ]:
t = pd.read_csv("archivo/resultados/tabla_por_conjunto.csv").set_index("sistema")
cols = {"annotation/lote1.csv BLEU": "lote base (213)", "annotation/lote_1_noticias.csv BLEU": "primera fase\nnoticias (72)", "annotation/lote_1_tuits.csv BLEU": "primera fase\ntuits (165)"}
sist = ["identidad", "reglas_equipo", "reglas_flo", "llm_zero_shot", "llm_dicc_flo", "mt5_flo"]
f1 = t.loc[[s for s in sist if s in t.index], list(cols)].rename(columns=cols).T
f1.columns = [NOMBRE.get(c, "mT5 (fase 1)" if c == "mt5_flo" else c) for c in f1.columns]
COLOR["mT5 (fase 1)"] = "#C9B3E3"
fig, ax = plt.subplots(figsize=(10, 4.2))
barras(ax, f1)
ax.set_ylim(75, 100)
ax.set_ylabel("BLEU")
ax.set_title("Fase 1: el LLM con diccionario es el mejor; el mT5 de entonces no supera a la identidad")
ax.legend(ncol=3, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.1), frameon=False)
guardar(fig, "fig1_fase1_bleu")

## 3. Fase 2: métricas por palabra

**ERR**: 0 es no cambiar nada; negativo es estropear más de lo que se arregla. Se grafican los cuatro sistemas del pipeline final en los cinco conjuntos.

In [ ]:
m = pd.read_csv("resultados/metricas_normalizador.csv")
m["sistema"] = m["sistema"].map(NOMBRE)
orden = ["reglas_equipo", "reglas_curado", "mT5 denso", "mT5 con MoE"]
conj = [("lote1", "lote base\n(213)"), ("lote_1_noticias", "primera fase\nnoticias (72)"), ("lote_1_tuits", "primera fase\ntuits (165)"),
        ("lote_1_2_noticias", "segunda fase\nnoticias (77)"), ("lote_1_2_tuits", "segunda fase\ntuits (308)")]
conj = [(k, n) for k, n in conj if k in set(m["conjunto"])]


def tabla_metrica(col):
    return pd.DataFrame({s: [m[(m.sistema == s) & (m.conjunto == k)][col].iloc[0] for k, _ in conj] for s in orden}, index=[n for _, n in conj])


err = tabla_metrica("err")
fig, ax = plt.subplots(figsize=(11, 4.6))
barras(ax, err)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("ERR (0 = no cambiar nada)")
ax.set_title("Solo las reglas con el diccionario revisado mejoran el texto; los mT5 quedan bajo 0")
ax.legend(ncol=4, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.14), frameon=False)
guardar(fig, "fig2_err")

f1m = tabla_metrica("f1")
fig, ax = plt.subplots(figsize=(11, 4.2))
barras(ax, f1m)
ax.set_ylabel("F1 de los cambios (%)")
ax.set_title("F1: los mT5 aprenden la tarea (26 a 29) pero no alcanzan a las reglas con el diccionario revisado")
ax.legend(ncol=4, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.12), frameon=False)
guardar(fig, "fig3_f1")

### Precisión contra recall

Cada punto es un sistema en un conjunto. Arriba a la derecha es mejor. Las reglas con el diccionario revisado aciertan más de lo que cambian (precisión de 46 a 56 %) pero cambian poco (recall de 22 a 34 %).

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5))
for s in orden:
    d = m[m.sistema == s]
    ax.scatter(d["recall"], d["precision"], s=60, color=COLOR[s], label=s, edgecolor="white")
ax.set_xlabel("Recall (%)")
ax.set_ylabel("Precisión (%)")
ax.set_xlim(0, 45)
ax.set_ylim(0, 65)
ax.set_title("Precisión y recall por sistema (un punto por conjunto)")
ax.legend(fontsize=8, frameon=False)
guardar(fig, "fig4_precision_recall")

## 4. Fase 2: qué pasa con cada oración

Para cada oración completa hay tres casos que no se solapan: **acierta** (queda igual a la referencia), **no cambia aunque debía** (queda igual a la original y la referencia era otra) y **cambia pero no acierta**. Se calcula con las predicciones del repo.

In [ ]:
def norm(s):
    s = unicodedata.normalize("NFKC", str(s)).casefold()
    return re.sub(r"\s+", " ", re.sub(r"[^\w@#\s]", " ", s)).strip()


leer = lambda p: pd.read_csv(p, encoding="utf-8-sig", keep_default_na=False, dtype=str)
mt, moe = leer("data/processed/predicciones_mt5_flo_fp32.csv"), leer("data/processed/predicciones_mt5_moe_fp32.csv")
filas = []
for clave, nombre in [("lote1", "lote base"), ("lote_1_tuits", "primera fase tuits"), ("lote_1_noticias", "primera fase noticias"), ("lote_1_2_tuits", "segunda fase tuits")]:
    ruta = f"data/processed/predicciones_{clave}.csv"
    if not os.path.exists(ruta):
        continue
    base = leer(ruta).drop_duplicates("chileno").set_index("chileno")
    a = mt[mt.conjunto == clave + ".csv"].set_index("chileno")
    b = moe[moe.conjunto == clave + ".csv"].set_index("chileno")
    idx = [i for i in a.index if i in base.index and i in b.index]
    salidas = {"reglas_equipo": base.loc[idx, "reglas_equipo"], "reglas_curado": base.loc[idx, "reglas_flo"],
               "mT5 denso": a.loc[idx, "mt5_flo_fp32"], "mT5 con MoE": b.loc[idx, "mt5_moe_fp32"]}
    for s, ser in salidas.items():
        ac = np.mean([norm(ser[i]) == norm(a.loc[i, "referencia"]) for i in idx])
        sc = np.mean([norm(ser[i]) == norm(i) and norm(ser[i]) != norm(a.loc[i, "referencia"]) for i in idx])
        filas.append({"conjunto": f"{nombre} ({len(idx)})", "sistema": s, "acierta": 100 * ac,
                      "no cambia aunque debía": 100 * sc, "cambia pero no acierta": 100 * (1 - ac - sc)})
o = pd.DataFrame(filas)
conjuntos = list(dict.fromkeys(o["conjunto"]))
fig, axs = plt.subplots(1, len(conjuntos), figsize=(3.1 * len(conjuntos), 3.8), sharey=True)
axs = np.atleast_1d(axs)
colores = {"acierta": "#6BAA75", "no cambia aunque debía": "#CFC7BF", "cambia pero no acierta": "#E07B6A"}
print(o.round(0).to_string())
for ax, c in zip(axs, conjuntos):
    d = o[o.conjunto == c].set_index("sistema").loc[orden]
    base_ = np.zeros(len(d))
    for col in colores:
        ax.bar(range(len(d)), d[col].clip(lower=0), bottom=base_, color=colores[col], label=col)
        base_ += d[col].clip(lower=0).values
    ax.set_xticks(range(len(d)))
    ax.set_xticklabels([s.replace(" ", "\n").replace("_", "\n") for s in d.index], fontsize=7)
    ax.set_title(c, fontsize=9)
axs[0].set_ylabel("% de oraciones")
axs[-1].legend(fontsize=7, loc="upper left", bbox_to_anchor=(1, 1), frameon=False)
fig.suptitle("Los mT5 cambian algo sin acertar en 56 a 65 % de las oraciones", y=1.02)
guardar(fig, "fig5_oraciones")

## 5. Fase 2: entrenamiento de los mT5

Pérdida de validación por época (8 épocas, 10.490 pares, una semilla). Fuente: registros de la corrida local del 25-nov-2026 (`logs/cadena_mt5_denso.log` y `cadena_mt5_moe.log`, que no se suben al repo). Los dos tocan su mínimo en la época 6; las predicciones usan esos pesos.

In [ ]:
denso = [0.698, 0.559, 0.491, 0.455, 0.434, 0.411, 0.428, 0.433]
moe_ = [0.656, 0.521, 0.453, 0.447, 0.425, 0.407, 0.414, 0.436]
ep = range(1, 9)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(ep, denso, "-o", color=COLOR["mT5 denso"], label="mT5 denso")
ax.plot(ep, moe_, "-o", color=COLOR["mT5 con MoE"], label="mT5 con MoE")
ax.axvline(6, color="gray", ls=":", lw=1)
ax.annotate("mejor época (6)", (6, 0.47), ha="center", fontsize=8, color="gray")
ax.set_xlabel("época")
ax.set_ylabel("pérdida de validación")
ax.set_title("Los dos mT5 empiezan a sobreajustar después de la época 6")
ax.legend(frameon=False)
guardar(fig, "fig6_perdida_validacion")

## 6. Fase 3: evaluación extrínseca

**Exact-Match Accuracy**: qué porcentaje de los términos ancla se traduce bien al inglés tras normalizar. Se compara el glosario final con los dos de control (sin la familia Claude y solo diccionarios).

In [ ]:
glos = {"glosario final": "resultados/eval_extrinseca.csv", "sin la familia Claude": "resultados/eval_extrinseca_sin_claude.csv",
        "solo diccionarios": "resultados/eval_extrinseca_solo_diccionarios.csv"}
ord5 = ["identidad", "reglas_equipo", "reglas_curado", "mT5 denso", "mT5 con MoE"]
fig, axs = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, (trad, nombre) in zip(axs, [("mt", "Opus-MT"), ("nllb", "NLLB")]):
    d = {}
    for g, ruta in glos.items():
        if os.path.exists(ruta):
            e = pd.read_csv(ruta)
            e = e[(e.conjunto == "todos") & (e.traductor == trad)].copy()
            e["sistema"] = e["sistema"].map(NOMBRE)
            d[g] = e.set_index("sistema")["ema"].reindex(ord5)
    tabla = pd.DataFrame(d)
    x = np.arange(len(tabla))
    n = len(tabla.columns)
    for i, (g, tono) in enumerate(zip(tabla.columns, ["#FF946C", "#F2B8A0", "#CFC7BF"])):
        b = ax.bar(x + (i - (n - 1) / 2) * 0.8 / n, tabla[g], 0.8 / n, label=g, color=tono)
        ax.bar_label(b, fmt="%.1f", fontsize=7, padding=1)
    ax.set_xticks(x)
    ax.set_xticklabels([s.replace(" ", "\n") for s in tabla.index], fontsize=8)
    ax.set_title(nombre)
axs[0].set_ylabel("Exact-Match Accuracy (%)")
axs[0].legend(fontsize=8, frameon=False)
fig.suptitle("Normalizar ayuda a traducir, y el orden de los sistemas no depende del glosario", y=1.02)
guardar(fig, "fig7_extrinseca")

## 7. Fase 3: el glosario por votos

**Calibración de la regla** con los 93 términos revisados por el integrante 5, con las 7 familias completas (tabla de la sección 4.3 del informe). Un umbral más estricto acepta menos términos pero sus glosas coinciden más con la referencia. Se eligió k = 5 antes de ver qué términos quedaban a cada lado.

In [ ]:
k = [3, 4, 5, 6, 7]
aceptados = [328, 313, 271, 213, 95]
respaldadas = [34, 47, 57, 62, 75]
fig, ax = plt.subplots(figsize=(6.5, 4))
b = ax.bar([str(i) for i in k], aceptados, color=["#CFC7BF", "#CFC7BF", "#FF946C", "#CFC7BF", "#CFC7BF"])
ax.bar_label(b, fontsize=8, padding=1)
ax.set_xlabel("familias que deben coincidir (de 7)")
ax.set_ylabel("términos con glosa aceptada (de 330)")
ax2 = ax.twinx()
ax2.plot([str(i) for i in k], respaldadas, "-o", color="#5B3E96")
ax2.set_ylabel("glosas respaldadas por la referencia (%)", color="#5B3E96")
ax2.set_ylim(0, 100)
ax2.spines["right"].set_visible(True)
ax.set_title("Más estricto: menos términos aceptados, glosas más confiables")
guardar(fig, "fig8_calibracion_votos")

# resultado de las dos muestras de términos aceptados por votos, revisadas por el integrante 1
mu = pd.DataFrame({"muestra 1 (20)": [15, 4, 1], "muestra 2 (40)": [31, 7, 2], "total (60)": [46, 11, 3]},
                  index=["ok sin cambios", "correcta pero incompleta", "distinta"])
pct = 100 * mu / mu.sum()
fig, ax = plt.subplots(figsize=(7, 3.4))
izq = np.zeros(3)
for fila, tono in zip(pct.index, ["#6BAA75", "#E8C468", "#E07B6A"]):
    ax.barh(pct.columns, pct.loc[fila], left=izq, color=tono, label=fila)
    for j, (v, c) in enumerate(zip(pct.loc[fila], mu.loc[fila])):
        if v > 6:
            ax.text(izq[j] + v / 2, j, f"{c}", ha="center", va="center", fontsize=8)
    izq += pct.loc[fila].values
ax.set_xlabel("% de los términos muestreados")
ax.set_title("57 de 60 términos aceptados por votos tenían glosas válidas (95 %)")
ax.legend(fontsize=8, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=3)
guardar(fig, "fig9_muestras_votos")

## 8. Quién revisó y posibles sesgos

**Revisión del diccionario por persona** (`data/curado/diccionario_flo.csv` y `terminos_descartados.csv`): los revisores no recibieron los mismos términos ni usaron los mismos criterios.

In [ ]:
dic = leer("data/curado/diccionario_flo.csv")
desc = leer("data/curado/terminos_descartados.csv")
ETIQ = {"Flo": "integrante\n1", "integrante 2": "integrante\n2", "integrante 3": "integrante\n3", "integrante 4": "integrante\n4",
        "Claude (validado por Flo)": "Claude\n(validado por 1)"}
dej = dic.groupby("reviso").size()
qui = desc.groupby("reviso").size()
amb = dic[dic.ambiguo == "1"].groupby("reviso").size()
quien = list(ETIQ)
dej, qui, amb = dej.reindex(quien).fillna(0), qui.reindex(quien).fillna(0), amb.reindex(quien).fillna(0)
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].bar([ETIQ[q] for q in quien], dej, color="#6BAA75", label="dejados en el diccionario")
axs[0].bar([ETIQ[q] for q in quien], qui, bottom=dej, color="#CFC7BF", label="quitados")
for i, q in enumerate(quien):
    axs[0].text(i, dej[q] + qui[q] + 15, f"{100 * qui[q] / (dej[q] + qui[q]):.0f} % quitados", ha="center", fontsize=8)
axs[0].set_ylabel("términos revisados")
axs[0].set_title("Cuántos términos revisó cada persona")
axs[0].legend(fontsize=8, frameon=False)
axs[1].bar([ETIQ[q] for q in quien], 100 * amb / dej, color="#9B6BC4")
for i, q in enumerate(quien):
    axs[1].text(i, 100 * amb[q] / dej[q] + 0.8, f"{100 * amb[q] / dej[q]:.0f} %", ha="center", fontsize=8)
axs[1].set_ylabel("% de los dejados marcados como ambiguos")
axs[1].set_title("Criterio de «ambiguo»: 37 % contra 0 a 8 %")
guardar(fig, "fig10_revisores_diccionario")

**¿Importa quién revisó el término ancla?** Exact Match por grupo de revisor del término, con Opus-MT. El integrante 1 revisó los términos más frecuentes, así que quién revisó y la frecuencia del término quedan mezclados: la brecha aparece también en los mT5, que no usan el diccionario al traducir.

In [ ]:
ev = leer("data/processed/eval_extrinseca.csv")
ev = ev[ev.exact_match != ""].drop_duplicates(["chileno", "sistema", "traductor"])
quien_term = dic.drop_duplicates("termino").set_index("termino")["reviso"]
ev["grupo"] = ev["termino"].map(quien_term).map(lambda r: "integrante 1" if r == "Flo" else ("integrantes 2, 3 y 4" if str(r).startswith("integrante") else "otros"))
ev["em"] = ev["exact_match"].astype(int)
sel = ev[(ev.traductor == "mt") & (ev.grupo.isin(["integrante 1", "integrantes 2, 3 y 4"]))].copy()
sel["sistema"] = sel["sistema"].map(NOMBRE)
tab = (sel.pivot_table(index="grupo", columns="sistema", values="em", aggfunc="mean") * 100)[ord5]
n = sel[sel.sistema == "identidad"].groupby("grupo").size()
tab.index = [f"{g} ({n[g]} oraciones)" for g in tab.index]
fig, ax = plt.subplots(figsize=(10, 4.2))
barras(ax, tab)
ax.set_ylabel("Exact-Match Accuracy (%), Opus-MT")
ax.set_title("La ventaja de las reglas sobre los mT5 es mayor en los términos de otros revisores")
ax.legend(ncol=5, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.1), frameon=False)
guardar(fig, "fig11_sesgo_revisor")

## 9. Listo

Las figuras quedan en `docs/figuras/` y se insertan en el informe con `![...](figuras/nombre.png)`.